#### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

/home/aniruddha/Projects/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
model = init_chat_model("openai:gpt-5-mini-2025-08-07")

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

In [3]:
model.invoke("What is LangChain?")

AIMessage(content='Short answer\n- LangChain is an open‑source framework for building applications that use large language models (LLMs). It provides reusable building blocks and integrations to make it easier to compose prompts, connect models, manage context/memory, retrieve documents, run multi‑step chains, and create agents that take actions.\n\nKey concepts (high level)\n- LLM wrappers: unified interfaces to call models (OpenAI, Anthropic, Hugging Face, local models, etc.).\n- Prompt templates: parameterized prompts with templating and prompt management.\n- Chains: composable sequences of steps (e.g., call LLM → parse → call another tool).\n- Agents & tools: agentic workflows where an LLM decides which tools (APIs, search, calculators, retrievers) to call and when.\n- Memory: persistent conversational/context memory across calls.\n- Document loaders & text splitters: ingest and chunk documents from files, web pages, PDFs, etc.\n- Embeddings & vector stores: create vector embedding

Use text prompts when:
- You have a single, standalone request
- You don’t need conversation history
- You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.


### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [4]:
from langchain.messages import SystemMessage, HumanMessage,AIMessage

messages = [
    SystemMessage("You are defence technology expert"),
    HumanMessage("Write difference between 4G and 5G technologies in fighter plane")
]

response = model.invoke(messages)

print(response.usage_metadata)
print(response.content)
print(response.response_metadata)


{'input_tokens': 29, 'output_tokens': 2377, 'total_tokens': 2406, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 1024}}
Short answer
- 4G (LTE) gives reliable IP connectivity and modest broadband capability useful for non-time‑critical ISR and C2 data on fighters.  
- 5G adds dramatically higher peak throughput, much lower latency, support for network slices and edge computing, and features for massive device density — enabling new time‑sensitive, data‑rich and distributed missions — but it requires hardened hardware, spectrum planning, and special engineering for high‑speed aircraft and contested EM environments.

Detailed comparison (fighter‑plane context)

1) Throughput / capacity
- 4G: Typical uplink/downlink tens-to-hundreds of Mbps in good conditions. Sufficient for compressed video, voice, telemetry.  
- 5G: Peak multi‑Gbps (especially mmWave). Enables multiple simultaneous high‑resolution video feeds, raw sensor data shar

In [5]:
## Detailed info to the LLM through System message
system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

Short answer: design your resources and endpoints, pick a web framework, implement routes that map HTTP verbs to operations (GET/POST/PUT/PATCH/DELETE), validate/serialize input and output, add auth, docs, tests, and deploy.

Below I’ll give a concise, practical example using FastAPI (recommended for new Python REST APIs: fast, async, automatic OpenAPI docs, Pydantic validation). I’ll explain key design choices and include a persistent example with SQLite, plus notes about auth, testing, and deployment.

1) Core design points (before coding)
- Model your resources (entities) and relationships.
- Map CRUD operations to HTTP verbs: GET (read), POST (create), PUT/PATCH (update), DELETE (delete).
- Use proper HTTP status codes (200/201/204/400/401/404/422/500).
- Validate and serialize with types (Pydantic).
- Add authentication/authorization (JWT/OAuth2), rate limiting, CORS, TLS.
- Provide docs/OpenAPI and versioning strategy (URL or header).
- Add automated tests and CI, and use migrati

In [6]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

response_human_msg = model.invoke([human_msg])

print(response_human_msg)
print(response_human_msg.response_metadata)

content='Hello! How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 10, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DxntFF2SfxLHWX58yNoKSylT4uZy3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f2bc1-10e4-7de1-ad06-5b6a1ae2725f-0' usage_metadata={'input_tokens': 10, 'output_tokens': 82, 'total_tokens': 92, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}}
{'token_usage': {'completion_tokens': 82, 'prompt_tokens': 10, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 

In [8]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.usage_metadata, 'end=')
print(response.content, 'end=')

{'input_tokens': 48, 'output_tokens': 17, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} end=
2 + 2 = 4. end=


In [9]:
from langchain.messages import AIMessage, ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [10]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [11]:
response

AIMessage(content="Right now in San Francisco it's sunny and 72°F (about 22°C). \n\nWant an hourly forecast, tomorrow's high/low, or anything else (wind, humidity)?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 238, 'prompt_tokens': 47, 'total_tokens': 285, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DxnwIYj8v5YBdhuzBNVqehe8Eq3WO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f2bc3-f2d1-70f2-b033-327143eea313-0', usage_metadata={'input_tokens': 47, 'output_tokens': 238, 'total_tokens': 285, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}})